# 02 - Data Preparation and Features

This notebook shows how we turned the data into modeling tables. Each row should contain one target value, the segment label, calendar context, weather, holiday information, and recent consumption history.


## Setup

We import the feature preparation functions from the project source code.


In [10]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass


import pandas as pd

from group5_energy.pipeline import (
    HALF_TARGET,
    DAILY_TARGET,
    add_history_lags,
    latest_clients,
    load_daily_history,
    load_daily_weather,
    load_half_hourly_history,
    load_holidays,
    load_hourly_weather,
    load_temperatures,
    prepare_daily_frame,
    prepare_half_hourly_frame,
)


## Building the feature tables

Here we load the cleaned history, weather, temperature, and holiday data. Then we create separate feature tables for the half-hourly and settings.

The half-hourly table is used for the 48 hour forecast, and the daily table is used for the one month forecast.


In [11]:
half_history = load_half_hourly_history()
daily_history = load_daily_history()
weather_hourly = load_hourly_weather()
temperatures = load_temperatures()
weather_daily = load_daily_weather()
holidays = load_holidays()

half_clients = latest_clients(half_history, "DateTime")
daily_clients = latest_clients(daily_history, "Date")

half_features = prepare_half_hourly_frame(half_history, weather_hourly, temperatures, holidays, half_clients)
daily_features = prepare_daily_frame(daily_history, weather_daily, holidays, daily_clients)
half_features = add_history_lags(half_features, HALF_TARGET, "half_hourly")
daily_features = add_history_lags(daily_features, DAILY_TARGET, "daily")

half_features.shape, daily_features.shape

((80796, 34), (1683, 33))

The feature creation step joins outside information to consumption history and adds time based variables. After this point, the rows are ready for model training and validation.

Note that the targets, lags, and time index will be different for the daily and half hourly horizons.


## Half-hourly features

This sample shows the columns used by the short term model. We inspect some rows before training just to confirm that the target, weather, holiday flag, time slot, and lag features are present together.


In [12]:
half_features[[
    "Acorn", "DateTime", "Conso_moy", "temperature", "temperature_half_hour",
    "is_holiday", "half_hour_slot", "lag_48", "lag_336", "rolling_48_mean"
]].head(10)

,Acorn,DateTime,Conso_moy,temperature,temperature_half_hour,is_holiday,half_hour_slot,lag_48,lag_336,rolling_48_mean
0,ACORN-E,2012-06-30 22:00:00,0.208288,13.61,13.890,0,44,NaN,NaN,NaN
1,ACORN-E,2012-06-30 22:30:00,0.194951,13.61,13.890,0,45,NaN,NaN,NaN
2,ACORN-E,2012-06-30 23:00:00,0.171547,13.48,13.890,0,46,NaN,NaN,NaN
3,ACORN-E,2012-06-30 23:30:00,0.151595,13.48,13.890,0,47,NaN,NaN,NaN
4,ACORN-E,2012-07-01 00:00:00,0.140109,13.44,13.890,0,0,NaN,NaN,NaN
5,ACORN-E,2012-07-01 00:30:00,0.136640,13.44,13.890,0,1,NaN,NaN,NaN
6,ACORN-E,2012-07-01 01:00:00,0.123180,13.25,13.890,0,2,NaN,NaN,NaN
7,ACORN-E,2012-07-01 01:30:00,0.110136,13.25,13.335,0,3,NaN,NaN,NaN
8,ACORN-E,2012-07-01 02:00:00,0.101755,12.28,12.780,0,4,NaN,NaN,0.154556
9,ACORN-E,2012-07-01 02:30:00,0.102435,12.28,12.500,0,5,NaN,NaN,0.148689


The half hourly rows include the current timestamp, the target value, temperature, holiday context, the half hour slot, and recent history.

The 48 step lag points to the previous day at the same time. The 336 step lag points to the previous week at the same time.


## Daily features

The daily model has fewer rows. We still keep weather, calendar, holiday, lag, and rolling average information.


In [13]:
daily_features[[
    "Acorn", "Date", "Conso_kWh", "temperatureMean", "is_holiday",
    "weekday", "lag_1", "lag_7", "rolling_7_mean"
]].head(10)

,Acorn,Date,Conso_kWh,temperatureMean,is_holiday,weekday,lag_1,lag_7,rolling_7_mean
0,ACORN-E,2012-07-01,8.501714,15.025,0,6,NaN,NaN,NaN
1,ACORN-E,2012-07-02,8.727650,17.120,0,0,8.501714,NaN,NaN
2,ACORN-E,2012-07-03,8.489889,18.695,0,1,8.727650,NaN,8.614682
3,ACORN-E,2012-07-04,8.173539,18.765,0,2,8.489889,NaN,8.573084
4,ACORN-E,2012-07-05,8.159257,16.740,1,3,8.173539,NaN,8.473198
5,ACORN-E,2012-07-06,8.343194,16.035,0,4,8.159257,NaN,8.410410
6,ACORN-E,2012-07-07,8.361754,16.525,0,5,8.343194,NaN,8.399207
7,ACORN-E,2012-07-08,8.988568,16.190,0,6,8.361754,8.501714,8.393857
8,ACORN-E,2012-07-09,8.627590,15.370,0,0,8.988568,8.727650,8.463407
9,ACORN-E,2012-07-10,8.420828,15.410,0,1,8.627590,8.489889,8.449113


The daily table keeps one row per ACORN and date. The one day lag captures yesterdays level, while the seven day lag and seven day rolling mean capture the weekly rhythm.

## Missing values after feature creation

Lag features create missing values at the beginning of each ACORN series (there is no earlier observation to refer to). We calculate missing rates to separate feature gaps from real data quality problems.

In [14]:
missing_half = half_features.isna().mean().sort_values(ascending=False).head(15)
missing_daily = daily_features.isna().mean().sort_values(ascending=False).head(15)
pd.DataFrame({"half_hourly_missing_rate": missing_half}).join(
    pd.DataFrame({"daily_missing_rate": missing_daily}), how="outer"
)

,half_hourly_missing_rate,daily_missing_rate
Acorn,0.000000,0.000000
Acorn_grouped,0.000000,NaN
Conso_kWh,NaN,0.000000
Conso_moy,0.000000,NaN
Date,0.000000,0.000000
DateTime,0.000000,NaN
WeatherHour,0.000000,NaN
apparentTemperatureMax,NaN,0.000000
dewPoint,0.000000,NaN
lag_1,0.000037,0.001783


The missing values are mainly caused by lag and rolling features at the start of each segment history. This was expected, and is not the same as a broken timestamp or missing measurement.